# Categorical Variables
---

In datasets with categorical variables normaly you will get an error if you try to plug these variables into most machine learning models in Python without preprocessing them first. In this tutorial, we'll compare three approaches that you can use to prepare your categorical data.

1. **Drop Categorical Variables**: The easiest approach to dealing with categorical variables is to simply remove them from the dataset. This approach will only work well if the columns did not contain useful information.

2. **Ordinal Encoding**: Ordinal encoding assigns each unique value to a different integer. This approach assumes an ordering of the categories: "Never" (0) < "Rarely" (1) < "Most days" (2) < "Every day" (3). For tree-based models (like decision trees and random forests), you can expect ordinal encoding to work well with ordinal variables.

3. **One-Hot Encoding**: One-hot encoding creates new bool columns indicating the presence (or absence) of each possible value in the original data. For example: `Color` is a categorical variable with three categories: "Red", "Yellow", and "Green". The corresponding one-hot encoding contains one column for each possible value with 1 or 0 depending on the original value in `Color`.\
\
In contrast to ordinal encoding, one-hot encoding does not assume an ordering of the categories. Thus, you can expect this approach to work particularly well if there is no clear ordering in the categorical data (e.g., "Red" is neither more nor less than "Yellow"). We refer to categorical variables without an intrinsic ranking as nominal variables.\
\
One-hot encoding generally **does not perform well** if the categorical variable takes on a large number of values (i.e., you generally won't use it for variables taking more than 15 different values).

## Example
---

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
data = pd.read_csv('../input/melbourne-housing-snapshot/melb_data.csv')

# Separate target from predictors
y = data.Price
X = data.drop(['Price'], axis=1)

# Divide data into training and validation subsets
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Drop columns with missing values (simplest approach)
cols_with_missing = [col for col in X_train_full.columns if X_train_full[col].isnull().any()] 
X_train_full.drop(cols_with_missing, axis=1, inplace=True)
X_valid_full.drop(cols_with_missing, axis=1, inplace=True)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
low_cardinality_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and 
                        X_train_full[cname].dtype in [ "object", "string" ] ]

# Select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]

# Keep selected columns only
my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

In [13]:
X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,u,S,Southern Metropolitan,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,h,SA,Western Metropolitan,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,h,S,Western Metropolitan,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,u,SP,Northern Metropolitan,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,h,S,Western Metropolitan,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


In [14]:
low_cardinality_cols

['Type', 'Method', 'Regionname']

### Define Function to Measure Quality of Each Approach
---

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

### Approach 1: Drop Categorical Variables
---

In [24]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
175703.48185157913


### Approach 2: Ordinal Encoding
---
Scikit-learn has a OrdinalEncoder class that can be used to get ordinal encodings. We loop over the categorical variables and apply the ordinal encoder separately to each column.

In [33]:
# s = (X_train.dtypes == 'object') # I also need string due to pandas 2.x.x

# Option A
# This check if dtype 'is in' the options list 
# s = X_train.dtypes.isin(['object', 'string'])

# Option B
# Using logical OR (I prefer it)
s = (X_train.dtypes == 'object') | (X_train.dtypes == 'string')

object_cols = list(s[s].index) # Categorical Variables

In [34]:
from sklearn.preprocessing import OrdinalEncoder

# Make copy to avoid changing original data 
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

# Apply ordinal encoder to each column with categorical data
ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
165936.40548390493


In [35]:
# Take a peek at the dataframe after apply OrdinalEncoder
label_X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,2.0,1.0,5.0,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,0.0,2.0,6.0,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,0.0,1.0,6.0,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,2.0,3.0,2.0,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,0.0,1.0,6.0,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


<details>
<summary>Ver <b>¿Que pasa aquí con el Orden de las Categorias?</b></summary>

De forma predeterminada, cuando no se pasa ninguna instrucción adicional, `OrdinalEncoder()` asigna los números basándose estrictamente en el orden alfabético (lexicográfico) de las categorías que encuentra durante el paso `.fit()`.

Esto tiene un impacto. Si se tiene una columna que evalúa el estado de una casa con los valores:
```python 
["Malo", "Regular", "Bueno"]
```
El codificador los ordenará alfabéticamente y les asignará:

- Bueno = 0
- Malo = 1
- Regular = 2

Acaba de destruir la jerarquía lógica de los datos.

### ¿Por qué el curso lo hace así y por qué parece no importar?
---
En este punto el curso estás trabajando con modelos basados en árboles (Decision Trees o Random Forests). Estos algoritmos son **extremadamente flexibles**; no interpretan esos números como una escala matemática estricta (no piensan que 2 vale el doble que 1), sino que simplemente los usan como "etiquetas numéricas" para agrupar casas y hacer divisiones lógicas (ej. "todas las casas con valor < 1 van a la izquierda"). Por eso, un orden alfabético arbitrario suele ser "suficientemente bueno" para que el árbol funcione.

Sin embargo, si estuvieras usando un modelo matemático estricto, como una **Regresión Lineal**, este orden alfabético arruinaría por completo las predicciones.

### ¿Cómo se establece el orden correcto?
---
En un proyecto real, cuando tú sabes que existe una jerarquía clara, no dejas que Scikit-Learn adivine. Le pasas tu propio orden explícito usando el parámetro `categories`, pasándole una lista con el orden exacto que deseas:

```python 
# Define the logical order yourself
orden_calidad = ['Malo', 'Regular', 'Bueno']

# It's passed to the encoder
ordinal_encoder = OrdinalEncoder(categories=[orden_calidad])

# Now Scikit-Learn set: Malo=0, Regular=1, Bueno=2
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
```

**Nota**: El curso advierte que obtendrás un mejor rendimiento si haces este trabajo manual. Esto ocurre porque le estás regalando al modelo un patrón matemático coherente. Cuando el algoritmo intenta predecir el precio de una casa, ahora entiende matemáticamente que "Siempre" (3) tiene un impacto progresivamente mayor que "Nunca" (0). Al no tener que crear divisiones extrañas en el árbol para desenredar un orden alfabético caótico, el modelo se vuelve más preciso y tu MAE baja.
</details>

### Approach 3: One-Hot Encoding
---
We use the OneHotEncoder class from scikit-learn to get one-hot encodings. There are a number of parameters that can be used to customize its behavior.

- We set `handle_unknown='ignore'` to avoid errors when the validation data contains classes that aren't represented in the training data, and
- setting `sparse_output=False` (previously called `sparse`) ensures that the encoded columns are returned as a numpy array (instead of a sparse matrix).

To use the encoder, we supply only the categorical columns that we want to be one-hot encoded. For instance, to encode the training data, we supply `X_train[object_cols]`. (`object_cols` in the code cell below is a list of the column names with categorical data, and so `X_train[object_cols]` contains all of the categorical data in the training set.)

<details>
<summary>check out <b>¿Why setting handle_unknown?</b></summary>
El parámetro handle_unknown='ignore' es un mecanismo de defensa vital para cuando tu modelo se enfrente a datos nuevos.

Cuando ejecutas el `fit` sobre `X_train`, el codificador aprende una lista estricta de categorías (por ejemplo, los vecindarios 'Norte' y 'Sur'). Si en tu `X_valid` (o en la vida real, cuando tu modelo esté en producción) aparece una casa ubicada en el vecindario 'Este', el codificador por defecto entrará en pánico y detendrá la ejecución porque nunca aprendió sobre ese vecindario durante el entrenamiento. Al configurar `ignore`, le das esta orden: "Si ves una categoría que no conoces, no rompas el programa. Simplemente asigna un `0` en las columnas de 'Norte' y 'Sur' para esa fila y sigue adelante".
</details>

In [39]:
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoder to each column with categorical data
# OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
# Recently 'sparse' name was changed to sparse_output in modern versions of scikit-learn
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

OH_cols_train = pd.DataFrame(
                    OH_encoder.fit_transform(X_train[object_cols]),
                    columns=OH_encoder.get_feature_names_out(object_cols)  # catch category names to the resulting columns
                )
OH_cols_valid = pd.DataFrame(
                    OH_encoder.transform(X_valid[object_cols]),
                    columns=OH_encoder.get_feature_names_out(object_cols)
                )

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 3 (One-Hot Encoding):
166089.4893009678


In [38]:
# OH_cols_train.head() # without .get_feature_names_out()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
12167,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6524,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8413,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2919,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6043,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [40]:
OH_cols_train.head() # with .get_feature_names_out() for PROD envs

,Type_h,Type_t,Type_u,Method_PI,Method_S,Method_SA,Method_SP,Method_VB,Regionname_Eastern Metropolitan,Regionname_Eastern Victoria,Regionname_Northern Metropolitan,Regionname_Northern Victoria,Regionname_South-Eastern Metropolitan,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria
12167,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6524,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8413,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2919,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6043,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## Conclusion: Which approach is best?
---
In this case, dropping the categorical columns (**Approach 1**) performed worst, since it had the highest MAE score. As for the other two approaches, since the returned MAE scores are so close in value, there doesn't appear to be any meaningful benefit to one over the other.

In general, one-hot encoding (**Approach 3**) will typically perform best, and dropping the categorical columns (**Approach 1**) typically performs worst, but it varies on a case-by-case basis.

Don't forget: the world is filled with categorical data.\
You will be a much more effective Machine Learning Engineer if you know how to deal with them.